In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

In [3]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [4]:
device = 'cuda'

In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
# X_test = scaler.transform(X_test)

X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)

y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)

In [6]:
train_data = TensorDataset(X_train, y_train)
valid_data = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=False)

In [11]:
class CancerClassifier(nn.Module):
    def __init__(self, n_features, n_units_l1, n_units_l2, dropout_rate):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, n_units_l1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(n_units_l1, n_units_l2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
        )
        self.output_layer = nn.Linear(n_units_l2, 1)
    
    def forward(self, X):
        deep = self.deep_stack(X)
        return self.output_layer(deep)

In [8]:
def train(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_train_loss = total_loss / len(train_loader)

        model.eval()
        total_valid_loss = 0
        total_correct = 0      # New: Keep track of right answers
        total_samples = 0      # New: Keep track of total patients tested
        
        with torch.no_grad():
            for X_batch, y_batch in valid_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                # 1. Get raw predictions and calculate loss
                raw_pred = model(X_batch)
                loss = criterion(raw_pred, y_batch)
                total_valid_loss += loss.item()
                
                # 2. ACCURACY MATH: Convert raw numbers to probabilities (0 to 1)
                probabilities = torch.sigmoid(raw_pred)
                
                # 3. Round to 0 or 1 (If > 0.5, guess 1. Else guess 0)
                guesses = probabilities.round()
                
                # 4. Count how many guesses match the actual y_batch
                correct_guesses = (guesses == y_batch).sum().item()
                
                total_correct += correct_guesses
                total_samples += len(y_batch) # Add the number of patients in this batch
                
        mean_valid_loss = total_valid_loss / len(valid_loader)
        
        # Calculate final accuracy percentage for the epoch
        accuracy = (total_correct / total_samples) * 100
        
        # Print the report!
        print(f"Epoch {epoch + 1:02d}/{n_epochs} | Train Loss: {mean_train_loss:.4f} | Valid Loss: {mean_valid_loss:.4f} | Accuracy: {accuracy:.2f}%")
    return accuracy

In [12]:
import optuna
def objective(trial):
    l1_neurons = trial.suggest_int("l1_neurons", 16, 64)
    l2_neurons = trial.suggest_int("l2_neurons", 4, 32)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)

    model = CancerClassifier(
        n_features=30,
        n_units_l1=l1_neurons,
        n_units_l2=l2_neurons,
        dropout_rate=dropout_rate
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    final_score = train(
        model=model,
        optimizer=optimizer,
        criterion=criterion,
        train_loader=train_loader,
        valid_loader=valid_loader,
        n_epochs=20
    )
    return final_score


In [13]:
study = optuna.create_study(direction="maximize")
print("Starting Optuna search...")

study.optimize(objective, n_trials=20)

print("\n---WINNING SETUP---")
print(f"Best Accuracy: {study.best_value:.2f}%")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-04-19 11:14:01,958] A new study created in memory with name: no-name-3d5c7dd1-a506-4418-aebd-229810c9e03c


Starting Optuna search...
Epoch 01/20 | Train Loss: 0.3107 | Valid Loss: 0.1023 | Accuracy: 96.49%
Epoch 02/20 | Train Loss: 0.1968 | Valid Loss: 0.1131 | Accuracy: 96.49%
Epoch 03/20 | Train Loss: 0.1068 | Valid Loss: 0.0653 | Accuracy: 97.37%
Epoch 04/20 | Train Loss: 0.0874 | Valid Loss: 0.0540 | Accuracy: 98.25%
Epoch 05/20 | Train Loss: 0.0871 | Valid Loss: 0.0473 | Accuracy: 98.25%
Epoch 06/20 | Train Loss: 0.0713 | Valid Loss: 0.0886 | Accuracy: 98.25%
Epoch 07/20 | Train Loss: 0.0668 | Valid Loss: 0.0783 | Accuracy: 98.25%
Epoch 08/20 | Train Loss: 0.0629 | Valid Loss: 0.0930 | Accuracy: 96.49%
Epoch 09/20 | Train Loss: 0.0500 | Valid Loss: 0.1814 | Accuracy: 96.49%
Epoch 10/20 | Train Loss: 0.2154 | Valid Loss: 0.0668 | Accuracy: 98.25%
Epoch 11/20 | Train Loss: 0.1661 | Valid Loss: 0.0899 | Accuracy: 97.37%
Epoch 12/20 | Train Loss: 0.0967 | Valid Loss: 0.1056 | Accuracy: 98.25%
Epoch 13/20 | Train Loss: 0.0733 | Valid Loss: 0.2204 | Accuracy: 96.49%


[I 2026-04-19 11:14:03,239] Trial 0 finished with value: 96.49122807017544 and parameters: {'l1_neurons': 49, 'l2_neurons': 8, 'dropout_rate': 0.2861483097270568, 'lr': 0.035214196070914826}. Best is trial 0 with value: 96.49122807017544.


Epoch 14/20 | Train Loss: 0.0924 | Valid Loss: 0.1158 | Accuracy: 97.37%
Epoch 15/20 | Train Loss: 0.0874 | Valid Loss: 0.0917 | Accuracy: 97.37%
Epoch 16/20 | Train Loss: 0.0724 | Valid Loss: 0.0614 | Accuracy: 98.25%
Epoch 17/20 | Train Loss: 0.0603 | Valid Loss: 0.1694 | Accuracy: 96.49%
Epoch 18/20 | Train Loss: 0.0462 | Valid Loss: 0.2252 | Accuracy: 96.49%
Epoch 19/20 | Train Loss: 0.0993 | Valid Loss: 0.2079 | Accuracy: 97.37%
Epoch 20/20 | Train Loss: 0.0756 | Valid Loss: 0.6436 | Accuracy: 96.49%
Epoch 01/20 | Train Loss: 0.3632 | Valid Loss: 0.0530 | Accuracy: 97.37%
Epoch 02/20 | Train Loss: 0.1248 | Valid Loss: 0.0829 | Accuracy: 96.49%
Epoch 03/20 | Train Loss: 0.1026 | Valid Loss: 0.0628 | Accuracy: 97.37%
Epoch 04/20 | Train Loss: 0.0901 | Valid Loss: 0.0566 | Accuracy: 97.37%
Epoch 05/20 | Train Loss: 0.0997 | Valid Loss: 0.0671 | Accuracy: 97.37%
Epoch 06/20 | Train Loss: 0.0563 | Valid Loss: 0.0841 | Accuracy: 96.49%
Epoch 07/20 | Train Loss: 0.0575 | Valid Loss: 0.06

[I 2026-04-19 11:14:04,025] Trial 1 finished with value: 98.24561403508771 and parameters: {'l1_neurons': 52, 'l2_neurons': 15, 'dropout_rate': 0.4963545968992481, 'lr': 0.018287921173288677}. Best is trial 1 with value: 98.24561403508771.


...

Epoch 13/20 | Train Loss: 0.0514 | Valid Loss: 0.0750 | Accuracy: 97.37%
Epoch 14/20 | Train Loss: 0.0406 | Valid Loss: 0.0940 | Accuracy: 97.37%
Epoch 15/20 | Train Loss: 0.0634 | Valid Loss: 0.1126 | Accuracy: 96.49%
Epoch 16/20 | Train Loss: 0.0946 | Valid Loss: 0.0940 | Accuracy: 98.25%
Epoch 17/20 | Train Loss: 0.0265 | Valid Loss: 0.1408 | Accuracy: 96.49%
Epoch 18/20 | Train Loss: 0.0170 | Valid Loss: 0.1204 | Accuracy: 97.37%
Epoch 19/20 | Train Loss: 0.0233 | Valid Loss: 0.1206 | Accuracy: 98.25%
Epoch 20/20 | Train Loss: 0.0158 | Valid Loss: 0.1653 | Accuracy: 96.49%
Epoch 01/20 | Train Loss: 0.6432 | Valid Loss: 0.5949 | Accuracy: 89.47%
Epoch 02/20 | Train Loss: 0.5728 | Valid Loss: 0.5006 | Accuracy: 94.74%
Epoch 03/20 | Train Loss: 0.4796 | Valid Loss: 0.3839 | Accuracy: 96.49%
Epoch 04/20 | Train Loss: 0.3580 | Valid Loss: 0.2634 | Accuracy: 97.37%
Epoch 05/20 | Train Loss: 0.2703 | Valid Loss: 0.1764 | Accuracy: 96.49%
Epoch 06/20 | Train Loss: 0.1999 | Valid Loss: 0.12

[I 2026-04-19 11:14:15,960] Trial 18 finished with value: 99.12280701754386 and parameters: {'l1_neurons': 32, 'l2_neurons': 26, 'dropout_rate': 0.355566788791562, 'lr': 0.0010272846605317932}. Best is trial 18 with value: 99.12280701754386.


Epoch 16/20 | Train Loss: 0.0666 | Valid Loss: 0.0563 | Accuracy: 98.25%
Epoch 17/20 | Train Loss: 0.0729 | Valid Loss: 0.0543 | Accuracy: 99.12%
Epoch 18/20 | Train Loss: 0.0563 | Valid Loss: 0.0542 | Accuracy: 99.12%
Epoch 19/20 | Train Loss: 0.0737 | Valid Loss: 0.0543 | Accuracy: 99.12%
Epoch 20/20 | Train Loss: 0.0641 | Valid Loss: 0.0543 | Accuracy: 99.12%
Epoch 01/20 | Train Loss: 0.6924 | Valid Loss: 0.6575 | Accuracy: 47.37%
Epoch 02/20 | Train Loss: 0.6315 | Valid Loss: 0.5874 | Accuracy: 74.56%
Epoch 03/20 | Train Loss: 0.5720 | Valid Loss: 0.5106 | Accuracy: 91.23%
Epoch 04/20 | Train Loss: 0.4904 | Valid Loss: 0.4250 | Accuracy: 95.61%
Epoch 05/20 | Train Loss: 0.4131 | Valid Loss: 0.3378 | Accuracy: 95.61%
Epoch 06/20 | Train Loss: 0.3197 | Valid Loss: 0.2634 | Accuracy: 96.49%
Epoch 07/20 | Train Loss: 0.2662 | Valid Loss: 0.2000 | Accuracy: 96.49%
Epoch 08/20 | Train Loss: 0.2120 | Valid Loss: 0.1550 | Accuracy: 96.49%
Epoch 09/20 | Train Loss: 0.1757 | Valid Loss: 0.12

[I 2026-04-19 11:14:16,621] Trial 19 finished with value: 99.12280701754386 and parameters: {'l1_neurons': 43, 'l2_neurons': 28, 'dropout_rate': 0.26879236213770047, 'lr': 0.0005949189475445446}. Best is trial 18 with value: 99.12280701754386.


Epoch 17/20 | Train Loss: 0.0886 | Valid Loss: 0.0632 | Accuracy: 98.25%
Epoch 18/20 | Train Loss: 0.0819 | Valid Loss: 0.0614 | Accuracy: 98.25%
Epoch 19/20 | Train Loss: 0.0768 | Valid Loss: 0.0602 | Accuracy: 99.12%
Epoch 20/20 | Train Loss: 0.0760 | Valid Loss: 0.0578 | Accuracy: 99.12%

---WINNING SETUP---
Best Accuracy: 99.12%
  l1_neurons: 32
  l2_neurons: 26
  dropout_rate: 0.355566788791562
  lr: 0.0010272846605317932


In [15]:
best_params = study.best_params

print("Building final model with these parameters")
print(best_params)

final_model = CancerClassifier(
    n_features=30,
    n_units_l1=best_params["l1_neurons"],
    n_units_l2=best_params["l2_neurons"],
    dropout_rate=best_params["dropout_rate"]
).to(device)

final_optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params["lr"])
criterion = nn.BCEWithLogitsLoss()

print("\n---TRAINING FINAL MODEL---")
final_accuracy = train(
    model=final_model,
    optimizer=final_optimizer,
    criterion=criterion,
    train_loader=train_loader,
    valid_loader=valid_loader,
    n_epochs=50
)

torch.save(final_model.state_dict(), "cancer_predictor.pth")
print("\nModel weigths saved to 'best_cancer_model.pth'")

Building final model with these parameters
{'l1_neurons': 32, 'l2_neurons': 26, 'dropout_rate': 0.355566788791562, 'lr': 0.0010272846605317932}

---TRAINING FINAL MODEL---
Epoch 01/50 | Train Loss: 0.6597 | Valid Loss: 0.6212 | Accuracy: 79.82%
Epoch 02/50 | Train Loss: 0.5834 | Valid Loss: 0.5181 | Accuracy: 90.35%
Epoch 03/50 | Train Loss: 0.4826 | Valid Loss: 0.3868 | Accuracy: 94.74%
Epoch 04/50 | Train Loss: 0.3570 | Valid Loss: 0.2657 | Accuracy: 94.74%
Epoch 05/50 | Train Loss: 0.2629 | Valid Loss: 0.1783 | Accuracy: 96.49%
Epoch 06/50 | Train Loss: 0.1884 | Valid Loss: 0.1263 | Accuracy: 97.37%
Epoch 07/50 | Train Loss: 0.1445 | Valid Loss: 0.0990 | Accuracy: 97.37%
Epoch 08/50 | Train Loss: 0.1294 | Valid Loss: 0.0844 | Accuracy: 97.37%
Epoch 09/50 | Train Loss: 0.1194 | Valid Loss: 0.0755 | Accuracy: 97.37%
Epoch 10/50 | Train Loss: 0.1039 | Valid Loss: 0.0708 | Accuracy: 97.37%
Epoch 11/50 | Train Loss: 0.0880 | Valid Loss: 0.0673 | Accuracy: 98.25%
Epoch 12/50 | Train Loss: